## Cars' Price Predictor using **XGBoosting**

### Libs importing

In [9]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

In [10]:
df = pd.read_csv('../Daten/autoscout24-germany-dataset.csv')
print(f"Original dataset shape: {df.shape}")

Original dataset shape: (46405, 9)


### Clean the Data

In [11]:
df_clean = df.dropna()
df_clean = df_clean[
    (df_clean['price'] > 500) & (df_clean['price'] < 100000) &
    (df_clean['mileage'] >= 0) & (df_clean['mileage'] < 500000) &
    (df_clean['hp'] > 0) & (df_clean['hp'] < 1000) &
    (df_clean['year'] >= 1990) & (df_clean['year'] <= 2023)
]

# Remove price outliers
Q1, Q3 = df_clean['price'].quantile([0.25, 0.75])
IQR = Q3 - Q1
df_clean = df_clean[(df_clean['price'] >= Q1 - 1.5*IQR) & (df_clean['price'] <= Q3 + 1.5*IQR)]
print(f"Cleaned dataset shape: {df_clean.shape}")

Cleaned dataset shape: (42530, 9)


### Feature Engineering (add a new feature)

In [12]:
df_features = df_clean.copy()

current_year = 2023
df_features['age'] = current_year - df_features['year']
df_features['mileage_per_year'] = df_features['mileage'] / (df_features['age'] + 1)
df_features['hp_per_year'] = df_features['hp'] / (df_features['age'] + 1)
df_features['log_mileage'] = np.log1p(df_features['mileage'])

df_features['age_category'] = pd.cut(df_features['age'], 
                                   bins=[0, 3, 7, 12, float('inf')],
                                   labels=['New', 'Recent', 'Mature', 'Old'])
df_features['mileage_category'] = pd.cut(df_features['mileage'], 
                                       bins=[0, 50000, 100000, 150000, 200000, float('inf')],
                                       labels=['VeryLow', 'Low', 'Medium', 'High', 'VeryHigh'])

### Encode categorical variables and split them into test and train data

In [13]:
label_encoders = {}
categorical_cols = ['make', 'model', 'fuel', 'gear', 'offerType', 'age_category', 'mileage_category']

df_model = df_features.copy()
for col in categorical_cols:
    le = LabelEncoder()
    df_model[col + '_encoded'] = le.fit_transform(df_model[col].astype(str))
    label_encoders[col] = le

# Select features
feature_columns = [
    'mileage', 'hp', 'year', 'age', 'mileage_per_year', 'hp_per_year', 'log_mileage',
    'make_encoded', 'model_encoded', 'fuel_encoded', 'gear_encoded', 'offerType_encoded',
    'age_category_encoded', 'mileage_category_encoded'
]

X = df_model[feature_columns]
y = df_model['price']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=37)

print(f"Training set: {X_train.shape}, Test set: {X_test.shape}")

Training set: (34024, 14), Test set: (8506, 14)
